In [2]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [3]:
from dotenv import load_dotenv, find_dotenv
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
import os
from llama_index.embeddings.openai import OpenAIEmbedding

# Load environment variables
load_dotenv(find_dotenv())  
embed_model = OpenAIEmbedding(model_name = os.environ.get("EMBED_MODEL"))

if embed_model is None:
    raise ValueError("EMBED_MODEL environment variable is not set!")

# Load documents
documents = SimpleDirectoryReader(input_files=["../data/Pmg_lds.md"]).load_data()



In [4]:
# merge into a single large document rather than the one document per page

from llama_index.core import Document

document = Document(text="\n\n".join([doc.text for doc in documents]))

In [5]:
from llama_index.core.node_parser import HierarchicalNodeParser

node_parser = HierarchicalNodeParser.from_defaults(chunk_sizes=[2048, 512, 128])

In [6]:
nodes = node_parser.get_nodes_from_documents([document])

In [7]:
len(nodes)

1671

In [8]:
nodes[30].text

'Activity: Personal or Companion Study (Page 110)\nStudy the following table. Think of times when you have experienced any of the feelings, thoughts, or impressions described in the passages below. As you study and gain experience, add other passages to this list. Think of how you can use these principles to help others feel and recognize the Spirit.\n\n| Scriptures | Principles\n| --- | --- |\n| D&C 6:23; 11:12-14; Romans 15:13; Galatians 5:22-23 | Gives feelings of love, joy, peace, patience, meekness, gentleness, faith, and hope. |\n| D&C 8:2-3 | Gives ideas in the mind, feelings in the heart. |\n| D&C 128:1 | Occupies the mind and presses on the feelings. |\n| Joseph Smith—History 1:11-12 | Helps scriptures have strong effect. |\n| D&C 9:8-9 | Gives good feelings to teach if something is true. | |\n| Alma 32:28; D&C 6:14-15; 1 Corinthians 2:9-11 | Enlightens the mind. |\n| Alma 19:6 | Replaces darkness with light. |\n| Mosiah 5:2-5 | Strengthens the desire to avoid evil and obey th

In [9]:
from llama_index.core.node_parser import get_leaf_nodes

leaf_nodes = get_leaf_nodes(nodes)
print(leaf_nodes[30].text)

They want to feel secure in a world of changing values. They want “peace in this world, and eternal life in the world to come” (D&C 59:23), but they are “kept from the truth because they know not where to find it” (D&C 123:12).
The gospel of Jesus Christ as restored through the Prophet Joseph Smith will bless their families, meet their spiritual needs, and help them fulfill their deepest desires. Although they may not know why, they need relief from feelings of guilt that come from mistakes and sins.


In [10]:
nodes_by_id = {node.node_id: node for node in nodes}

parent_node = nodes_by_id[leaf_nodes[30].parent_node.node_id]
print(parent_node.text)

Consider This (Page 15)

- What is my purpose as a missionary?
- What is the gospel?
- Why do we preach the gospel?
- Why must I teach with power and authority?
- What is the message of the Restoration? Why is it so important?
- What is my responsibility in helping others become converted?
- How will I know whether I am a successful missionary?
  
> Questions to help study this chapter by looking for the answers





Your Commission to Teach the Restored Gospel of Jesus Christ (Page 15)

You are surrounded by people. You pass them on the street, visit them in their homes, and travel among them. All of them are children of God, your brothers and sisters. God loves them just as He loves you. Many of these people are searching for purpose in life. They are concerned for their families. They need the sense of belonging that comes from the knowledge that they are children of God, members of His eternal family. They want to feel secure in a world of changing values. They want “peace in this 

In [52]:
from llama_index.llms.openai import OpenAI

# llm = OpenAI(model ="gpt-3.5-turbo", temperature=0.1)
llm = OpenAI(model="gpt-4o-mini", temperature=0)

In [53]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model=embed_model
# Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.node_parser = node_parser

In [54]:
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core import load_index_from_storage

if not os.path.exists("../VectorStore"):
    os.makedirs("../VectorStore")
    storage_context = StorageContext.from_defaults()
    storage_context.docstore.add_documents(nodes)

    automerging_index = VectorStoreIndex(
        leaf_nodes, storage_context=storage_context)

    automerging_index.storage_context.persist(persist_dir="../VectorStore")

else:
    automerging_index = load_index_from_storage(StorageContext.from_defaults(persist_dir="../VectorStore"))

In [55]:
from llama_index.postprocessor.cohere_rerank import CohereRerank
load_dotenv(find_dotenv())
cohere_rerank = CohereRerank(
    api_key=os.environ["COHERE_API_KEY"], 
    top_n=6,
)

In [56]:
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

automerging_retriever = automerging_index.as_retriever(similarity_top_k=12)

retriever = AutoMergingRetriever(
    automerging_retriever,
    automerging_index.storage_context,
    verbose= True
)

auto_merging_engine = RetrieverQueryEngine.from_args(
    retriever, node_postprocessors=[cohere_rerank]
)

#### 4o-mini

In [57]:
response = auto_merging_engine.query("suggest some bullets points I can use to begin teaching when I eventually find somebody to teach?")

In [58]:
print(response)

- Start your first visit in a warm, respectful, and genuine manner to build trust.
- Ask simple questions about their religious background and expectations, such as, “What role has religion played in your life?”
- Encourage everyone present to join in the lesson and minimize distractions by turning off the television.
- Offer to begin and end each lesson with a prayer, asking for blessings and for them to feel the truth of your teachings.
- Quickly refer to the Restoration of the gospel, highlighting its significance as your unique message.
- Work with local leaders to identify individuals who may be open to your message, such as those who have recently experienced significant life events.
- Look for opportunities to provide simple acts of service to build connections.
- Make brief, daily visits to support and help those you are teaching, explaining your purpose during these visits.
- Keep detailed notes in your planner to follow up on commitments and invitations extended during lesson

### Evaluation

In [84]:
from trulens_eval import Tru

tru = Tru()
tru.reset_database()


In [85]:
import numpy as np
from trulens.apps.llamaindex import TruLlama
from trulens.core import Feedback
from trulens.providers.openai import OpenAI

# Initialize provider class
provider = OpenAI(model_engine="gpt-4o-mini")

# select context to be used in feedback. the location of context is app specific.

context = TruLlama.select_context(auto_merging_engine)

# Define a groundedness feedback function
f_groundedness = (
    Feedback(
        provider.groundedness_measure_with_cot_reasons, name="Groundedness"
    )
    .on(context.collect())  # collect context chunks into a list
    .on_output()
)

# Question/answer relevance between overall question and answer.
f_answer_relevance = Feedback(
    provider.relevance_with_cot_reasons, name="Answer Relevance"
).on_input_output()
# Question/statement relevance between question and each context chunk.
f_context_relevance = (
    Feedback(
        provider.context_relevance_with_cot_reasons, name="Context Relevance"
    )
    .on_input()
    .on(context)
    .aggregate(np.mean)
)

✅ In Groundedness, input source will be set to __record__.app.query.rets.source_nodes[:].node.text.collect() .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input context will be set to __record__.app.query.rets.source_nodes[:].node.text .


In [86]:
tru_query_engine_recorder = TruLlama(
    auto_merging_engine,
    app_name="LlamaIndex_App",
    app_version="Advanced_Retriever",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance],
)

In [87]:
import pandas as pd

# Read the CSV file and drop the 'Unnamed: 0' column
df = pd.read_csv("../dataset_eval/20_dataset.csv").drop(columns=['Unnamed: 0'])

# Extract the first 20 questions into a list
questions = df['question'][:20].tolist()

# Print the list of questions
print(questions)


['What strategies can be used to make a message easy to understand when teaching?', 'What should teachers do with unfamiliar words to ensure their message is easy to understand?', 'What are some effective study techniques to enhance understanding and retention of material?', 'What is the purpose of using a study journal in your scripture study?', 'What is the importance of organizing and summarizing lesson plans for effective teaching?', 'What is the recommended method for highlighting key words when marking scriptures?', 'What is the significance of beginning study activities with a prayer?', 'What is the significance of the power of ordination in the context of missionary work?', 'What is the purpose of marking scriptures in relation to applying gospel teachings?', 'What is the significance of preaching the gospel according to President Lorenzo Snow?', 'What is the relationship between the Atonement and missionary work according to President Howard W. Hunter?', 'What are the blessing

In [88]:
# evaluate
for question in questions:
    with tru_query_engine_recorder as recording:
        response = auto_merging_engine.query(question)

> Merging 2 nodes into parent node.
> Parent node id: 828b8306-9ab2-4ede-9c41-eb1dafcc8aab.
> Parent node text: Search, Ponder, and Remember (Page 36)
- Be alert. Exercise, shower, and pray before you study.
-...

> Merging 3 nodes into parent node.
> Parent node id: 63b7ba91-684f-4b9b-b13f-7004075fbda2.
> Parent node text: Use Study Resources (Page 37)
- Use the study aids in the LDS edition of the scriptures (Topical ...

> Merging 3 nodes into parent node.
> Parent node id: 4a11252f-bcc1-40a8-8ff2-ff1081645219.
> Parent node text: Marking Scriptures (Page 38)

Marking your scriptures can assist you in thinking deeply about a p...

> Merging 3 nodes into parent node.
> Parent node id: 63b7ba91-684f-4b9b-b13f-7004075fbda2.
> Parent node text: Use Study Resources (Page 37)
- Use the study aids in the LDS edition of the scriptures (Topical ...

> Merging 3 nodes into parent node.
> Parent node id: 828b8306-9ab2-4ede-9c41-eb1dafcc8aab.
> Parent node text: Search, Ponder, and Remember (Pa

In [89]:
records, feedback = tru.get_records_and_feedback(app_ids=[])
# records.head()

In [90]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
records[["input", "output"] + feedback]

,input,output,Context Relevance,Answer Relevance,Groundedness
0,"""What strategies can be used to make a message easy to understand when teaching?""","""To make a message easy to understand when teaching, consider the following strategies:\n\n1. **Define Unfamiliar Words**: Identify and explain any words or phrases that may be unfamiliar to your audience. Use simple definitions and refer to resources like the Bible Dictionary and True to the Faith for clarity.\n\n2. **Engage with Questions**: Ask your audience questions to gauge their understanding, such as, \u201cDo you know what we mean when we say...?\u201d This encourages interaction and helps clarify any confusion.\n\n3. **Observe Nonverbal Communication**: Pay attention to body language, facial expressions, and tone of voice, both yours and that of your audience. This can provide insights into their feelings and understanding.\n\n4. **Use Concise Statements**: Be prepared to present your message in a brief and impactful manner. Aim for short, powerful statements that convey the core message effectively.\n\n5. **Revisit Previous Lessons**: If necessary, return to earlier lessons to reinforce and clarify important doctrines, ensuring that your audience has a solid understanding.\n\nBy implementing these strategies, you can enhance comprehension and facilitate a more effective teaching experience.""",0.777778,1.0,0.666667
1,"""What should teachers do with unfamiliar words to ensure their message is easy to understand?""","""Teachers should take the time to define unfamiliar words and principles clearly. They can prepare by studying key definitions and terms, using resources like the Bible Dictionary and True to the Faith. It's important to explain these words simply and to check for understanding by asking questions. Teachers should also be attentive to whether a word or principle is new to their audience and be willing to revisit previous lessons to clarify any difficult concepts. Writing definitions in a study journal and sharing them with others can help refine their explanations. Ultimately, the goal is to ensure that both the teacher and the learners are edified and can rejoice together in their understanding.""",0.944444,1.0,1.000000
2,"""What are some effective study techniques to enhance understanding and retention of material?""","""To enhance understanding and retention of material, consider the following techniques:\n\n1. Begin with prayer to seek understanding.\n2. Study in a focused environment, such as at a desk or table, to remain alert.\n3. Organize your study materials and write notes to reinforce learning.\n4. Study by topic and ask reflective questions about the material.\n5. Visualize concepts to create a mental image of what you are learning.\n6. Engage with the words of living prophets and utilize approved resources.\n7. Keep a study journal to record questions and insights, and seek answers through scriptures and other resources.\n8. Mark your scriptures and take notes in the margins to clarify ideas.\n9. Share your learnings with others, as teaching can deepen your understanding.\n10. Memorize key passages that support the principles you are studying.\n11. Use new vocabulary in sentences and practice pronunciation to reinforce learning.\n12. Regularly review and revisit material to strengthen retention.""",0.611111,1.0,0.486111
3,"""What is the purpose of using a study journal in your scripture study?""","""The purpose of using a study journal in scripture study is to help understand, clarify, and remember what is being learned. It serves as a tool to record thoughts and impressions, allowing for the recall of spiritual experiences, recognition of growth, and the ability to access key information in the future. Additionally, it enhances the likelihood of receiving further insights and light, making it a valuable resource for personal and companion study, as well as for teaching and applying learned principles.""",0.722222,1.0,1.000000
4,"""What is th

In [91]:
tru.get_leaderboard()

,,Context Relevance,Answer Relevance,Groundedness,latency,total_cost
app_name,app_version,,,,,
LlamaIndex_App,Advanced_Retriever,0.697222,1.0,0.916435,13.35,0.000195


#### 3.5 turbo

In [42]:
response = auto_merging_engine.query("what scriptures can help me teach about faith?")

In [43]:
print(response)

1 Nephi 7:12, 2 Nephi 9:23, 2 Nephi 25:23, Moroni 7:33–34, Moroni 10:7, Alma 32, Ephesians 2:8, Ether 12, Hebrews 11, 1 Nephi 3:7, James 2:17–26, 2 Nephi 25:29, 2 Nephi 26:13, Mosiah 4:6–12, Helaman 15:7–8, and Ether 12:7–22.


In [45]:
response = auto_merging_engine.query("suggest some bullets points I can use to begin teaching when I eventually find somebody to teach?")

In [46]:
print(response)

- Begin with a warm, respectful, and genuine manner.
- Ask simple questions to understand their religious background and expectations.
- Encourage all present to join in the lesson and remove distractions.
- Start and end each lesson with a prayer.
- Refer quickly to the Restoration of the gospel.
- Offer brief, daily visits to support and help them.
- Make specific notes to follow up on commitments.
- Make frequent contact to check progress, answer questions, and provide additional teachings.


In [50]:
response = auto_merging_engine.query("How do i find people to teach?")


> Merging 3 nodes into parent node.
> Parent node id: a8db3ed9-bb96-4c70-9ad8-4a7e5ae24eaf.
> Parent node text: Activity: Companion Study (Page 180)
- Identify all the former investigators in your area book.
-...



In [51]:
print(response)

You can find people to teach by working with the bishop and ward council to identify individuals who have recently had significant life events, offering simple service opportunities, teaching members about the message of the Restoration and asking for referrals, organizing member firesides, teaching English as a second language, inviting people to Church meetings and activities, using pass-along cards and other materials, seeking referrals from various sources, coordinating with public affairs representatives for missionary service opportunities, and inviting people to baptismal services. Remember to seek the guidance of the Holy Ghost and be open to unplanned finding opportunities.
